## Search evaluation with ground truth

In [2]:
# For search evaluation, we only need the search part of the RAG pipeline. We don't need to call the LLM yet.

# Load the ground truth file from the previous notebook:

import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

# Use the same ingest.py file we downloaded in the previous notebook.

# Load the documents and build a minsearch index:

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)



In [4]:
# Wrap the search call in a function called text_search. The name is deliberate. 
# Later we'll write vector_search or a hybrid version and run the exact same evaluation on it. 
# Everything downstream only needs a function that takes a query and returns results, so we can swap one for another. 
# That mirrors how RAG works: the retrieval step doesn't care which search function sits behind it.

def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )


In [5]:
# Collecting relevance data
# Start with one ground truth record:

q = ground_truth[0]
q

# Run search for this question:

doc_id = q["document"]
results = text_search(query=q["question"])

# First, compare the retrieved document IDs with the correct document ID:

for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
69d122f12e == 74eb249bbf: False


In [6]:
# Then turn this comparison into a relevance list. In this lesson, relevance means whether a retrieved document is 
# the correct document for this question.

relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

# This gives a list of 0 and 1 values. 1 means the retrieved document has the same ID as the correct document.

# Put this logic into a function:

def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

    

In [7]:
# For the first ground truth record, the relevance list is:

q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

# The correct document was the first search result.

Can I still join the course if I just found it now?


[1, 0, 0, 0, 0]

In [8]:
# For this question:

q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

# Here, the correct document was also the first search result.

Where is the live stream link for the workshop or Office Hours usually announced before it starts?


[1, 0, 0, 0, 0]

In [9]:
# For this question:

q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

# The correct document was found at the first position again.

Where do I find the LLM Zoomcamp course page with the syllabus and deadlines?


[1, 0, 0, 0, 0]

In [10]:
# Now do the same thing for all ground truth questions:

from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total
    
# Call it for the first 15 ground truth questions:

ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)


  0%|          | 0/15 [00:00<?, ?it/s]

In [11]:
# Look at the results:

relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [12]:
# Next, make the relevance functions generic. We start with text search, but later we may want to evaluate vector search, 
# hybrid search, or another retrieval method. The relevance logic is the same. Only the search function changes.

def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance
    
# The total relevance function gets a search_function too.

In [13]:
# We need to provide it explicitly:

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total
    
# Use it with text_search on the same sample:

relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [14]:
# Now run it for all ground truth questions:

relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

## Search Evaluation Metrics

In [15]:
# Hit Rate
# Hit Rate (also called Recall@k) measures the fraction of queries where the correct document appears anywhere in the results:

example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [16]:
# Each line is one query. If a line contains 1, search found the correct document somewhere in the top 5 results. 
# If the line contains only zeros, search did not find the correct document.

# In our setup, each query has one correct document, so Hit Rate and Recall@k mean the same thing.

# Let's calculate it:

cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

# There are 14 hits. The example has 15 queries.

14

In [17]:
# The Hit Rate is:

cnt / len(example)
# 0.933
# This means that search found the correct document for 93.3% of the queries in this example.

0.9333333333333333

In [18]:
# Put the same logic into a function:

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)
    
# Check it on the same example:

hit_rate(example)
# 0.933

0.9333333333333333

In [19]:
# Mean Reciprocal Rank (MRR)
# Hit Rate tells us if we found the right document, but not where it was.

# MRR also considers the position.

# For each query, the score is based on the rank of the first correct document:

# position 1: score is 1.0
# position 2: score is 0.5
# position 3: score is 0.333
# not found: score is 0
# In the example, most hits are at the first position. Some hits are lower in the list.

# Look at that line:

example[1]
# [0, 1, 0, 0, 0]
# For this line, the score is 1/2 because the correct document is at position 2.

[0, 1, 0, 0, 0]

In [20]:
# Let's calculate MRR:

total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

# The total score is 12.333333333333334. We use rank + 1 because Python counts positions from zero. 
# The first position should score 1/1, and without the + 1 we'd divide by zero.

# Divide it by the number of queries:

total_score / len(example)
# 0.822
# MRR is the average of these scores across all queries. It rewards systems that put the correct document near the top.

0.8222222222222222

In [21]:
# Hit Rate is the upper bound for MRR. In practice, MRR is usually smaller because some correct documents are found below the first position.

# Put the same logic into a function:

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)
    
# Check it on the same example:

mrr(example)
# 0.822

0.8222222222222222

In [22]:
# Putting it together
# Wrap the metrics in a reusable evaluation function:

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }
    
# We can evaluate any search function:

evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.8442477876106195, 'mrr': 0.7162831858407077}

## Search Parameter Tuning

In [23]:
# In the previous lesson, we defined Hit Rate, MRR, and the evaluate function. Now we can use them to tune search parameters.

# Instead of guessing which settings are better, we measure them on the ground truth dataset.

# So far we've boosted question to 3.0. The idea was that a query should match the FAQ question. 
# That kind of match should count for more than matching the answer text. It sounds reasonable. 
# But it's a guess, and now we can check it against data instead of trusting it.

# This is the main benefit of offline evaluation. We change one parameter, run the same questions again, 
# and see whether the metric moves. The dataset stays fixed, so the comparison is fair.

# Trying different boosts
# Start with a search function where the question boost is configurable:

def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )
    
# Evaluate several boost values:

for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")
    
# For the data we prepared on May 29, 2026, this gives:

# boost=0.5: {'hit_rate': 0.9113924050632911, 'mrr': 0.800548523206751}
# boost=1.0: {'hit_rate': 0.9240506329113924, 'mrr': 0.8139240506329113}
# boost=3.0: {'hit_rate': 0.8987341772151899, 'mrr': 0.7693248945147676}
# boost=5.0: {'hit_rate': 0.8708860759493671, 'mrr': 0.7401265822784809}
# boost=10.0: {'hit_rate': 0.8582278481012658, 'mrr': 0.7122362869198313}

# Increasing the question boost makes the metrics worse, not better. The best value here is 1.0, no boost at all. 
# That's already the opposite of what the intuition predicted.

# But this is only one parameter. We can also tune answer and section together with question.

# Define a search function with all three boosts:

def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

  0%|          | 0/565 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8884955752212389, 'mrr': 0.7969616519174038}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8902654867256637, 'mrr': 0.7878171091445423}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8442477876106195, 'mrr': 0.7162831858407077}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8070796460176991, 'mrr': 0.6850442477876103}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7752212389380531, 'mrr': 0.6497640117994098}


In [24]:
# Now do a small grid search:

results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })
            
# Sort by MRR:

df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)


Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.968142,0.876460
19,2.0,4.0,0.2,0.968142,0.876460
35,5.0,10.0,0.5,0.968142,0.876460
6,1.0,4.0,0.1,0.966372,0.875929
34,5.0,10.0,0.2,0.966372,0.875428
18,2.0,4.0,0.1,0.966372,0.875428
33,5.0,10.0,0.1,0.966372,0.875192
7,1.0,4.0,0.2,0.966372,0.873864
4,1.0,2.0,0.2,0.966372,0.873156
22,2.0,10.0,0.2,0.966372,0.872596


In [25]:
# For the same data, the best rows are:

# question  answer  section  hit_rate  mrr
# 1.0       2.0     0.1      0.975     0.885
# 2.0       4.0     0.2      0.975     0.885
# 5.0       10.0    0.5      0.975     0.885
# 5.0       10.0    0.2      0.975     0.884
# 5.0       10.0    0.1      0.975     0.884
# 2.0       4.0     0.1      0.975     0.884
# 2.0       4.0     0.5      0.977     0.884
# 1.0       2.0     0.2      0.977     0.884
# 1.0       2.0     0.5      0.965     0.862
# 1.0       4.0     0.1      0.970     0.862

# The best combination weights answer twice as heavily as question, with almost no weight on section. 
# So the data says the opposite of where we started. The answer text matters more for retrieval than the question text. 
# The intuition was wrong, and we'd never have known without measuring it. This is exactly why we evaluate instead of guess.

# The first three rows have the same relative weights:

# question : answer : section = 1 : 2 : 0.1

# So we can use the smaller and easier-to-read values: question=1.0, answer=2.0, and section=0.1. 
# This gives the same relative weights as question=5.0, answer=10.0, and section=0.5, but the numbers are not unnecessarily large.

# Define the search function with these boosts:

def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )
    
# Usually we care about both metrics. Hit Rate tells us whether the correct document appears at all. 
# MRR tells us whether it appears near the top. A document near the top is more likely to be used by the RAG prompt.


In [26]:
## 